# Python code for project

Activity classification with Random Forest, XGBoost and LightGBM.

Needs `activity_train.csv` and `activity_test.csv` in the same folder as this notebook.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

## Load datasets

In [ ]:
# Load datasets
train_df = pd.read_csv("activity_train.csv")  # Adjust path as needed
test_df = pd.read_csv("activity_test.csv")

## Class distribution

In [ ]:
# Class distribution
train_class_counts = train_df['activity'].value_counts().sort_index()
test_class_counts = test_df['activity'].value_counts().sort_index()

# Plot class distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.barplot(x=train_class_counts.index, y=train_class_counts.values, palette="viridis")
plt.title("Training Set Activity Class Distribution")
plt.xlabel("Activity Class")
plt.ylabel("Count")

plt.subplot(1, 2, 2)
sns.barplot(x=test_class_counts.index, y=test_class_counts.values, palette="plasma")
plt.title("Test Set Activity Class Distribution")
plt.xlabel("Activity Class")
plt.ylabel("Count")

plt.tight_layout()
plt.show()

## Missing values and feature variance

In [ ]:
X = train_df.drop('activity', axis=1)
print("Missing values:", X.isnull().sum().sum())
print("Variance summary:\n", X.var().describe())

## Features, scaling and train/validation split

In [ ]:
# Separating features
X_full, y_full = train_df.drop('activity', axis=1), train_df['activity']
X_test, y_test = test_df.drop('activity', axis=1), test_df['activity']

# Standard scaling
scaler = StandardScaler()
X_full_scaled = scaler.fit_transform(X_full)
X_test_scaled = scaler.transform(X_test)

# Stratified split: 80/20
X_train, X_val, y_train, y_val = train_test_split(
    X_full_scaled, y_full,
    test_size=0.2,
    stratify=y_full
)

print("Shapes → Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test_scaled.shape)

## Random Forest

In [ ]:
# Random forest
rf = RandomForestClassifier()
rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)
print("RF Accuracy:", accuracy_score(y_val, y_pred))
print("RF Macro-F1:", f1_score(y_val, y_pred, average='macro'))
print(classification_report(y_val, y_pred, zero_division=0))

cm = confusion_matrix(y_val, y_pred)
plt.figure(figsize=(5,4))
plt.imshow(cm, cmap="Blues", interpolation='nearest')
plt.title("RF Confusion Matrix (Validation)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.colorbar()
plt.show()

## XGBoost

In [ ]:
# XGBOOST
from xgboost import XGBClassifier

# 1) Initialize XGBoost without the deprecated parameter.
# Explicitly set objective and number of classes.
xgb = XGBClassifier(
    objective="multi:softprob",
    num_class=12,
    eval_metric="mlogloss",
    verbosity=0  # suppress training-time messages
)

# 2) Fit on your already-encoded labels y_train_enc
xgb.fit(X_train, y_train_enc)

# 3) Predict on validation set (encoded labels)
y_val_pred_enc = xgb.predict(X_val)

# 4) Invert encoding for human-readable labels
y_val_pred = le.inverse_transform(y_val_pred_enc)

# 5) Evaluate
print("XGB Accuracy:", accuracy_score(y_val, y_val_pred))
print("XGB Macro-F1:", f1_score(y_val, y_val_pred, average="macro"))
print(classification_report(y_val, y_val_pred, zero_division=0))

# 6) Confusion matrix
cm = confusion_matrix(y_val, y_val_pred, labels=np.arange(1,13))
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens")
plt.title("XGBoost Confusion Matrix (Validation)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

## LightGBM

In [ ]:
# LGBM xD
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier()
lgbm.fit(X_train, y_train)

y_pred = lgbm.predict(X_val)
print("LGBM Accuracy:", accuracy_score(y_val, y_pred))
print("LGBM Macro-F1:", f1_score(y_val, y_pred, average='macro'))

cm = confusion_matrix(y_val, y_pred)
plt.figure(figsize=(5,4))
plt.imshow(cm, cmap="Oranges", interpolation='nearest')
plt.title("LightGBM Confusion Matrix (Validation)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.colorbar()
plt.show()